In [4]:
from __future__ import annotations

import math
from typing import List

import pandas as pd
from scipy.stats import pearsonr, spearmanr


def make_dataframe() -> pd.DataFrame:
    """
    Fill these values with your category-level results.

    Synthetic metrics:
      - use relative differences between synthetic and real spans
      - signed values are okay
      - the script can also derive absolute differences

    Model metrics:
      - choose one model summary per category
      - or best model per category
      - or average encoder performance per category
      - just be consistent
    """
    data = [
        {
            "category": "Format",
            "avg_span_len_diff": -7.1,
            "mattr_diff": -0.8,
            "ead2_diff": -20.4,
            "top20_diff": -3.9,
            "vocab_per_span_diff": -47.8,
            "f1": 0.929,
            "kappa": 0.788,
        },
        {
            "category": "Unsupported Claim (long)",
            # replace with your short-UC synthetic metrics if अलग from combined UC
            "avg_span_len_diff": -11.5,
            "mattr_diff": -0.8,
            "ead2_diff": -25.0,
            "top20_diff": 23.9,
            "vocab_per_span_diff": -49.3,
            "f1": 0.495,
            "kappa": 0.209,
        },
        {
            "category": "Coherence",
            "avg_span_len_diff": 31.6,
            "mattr_diff": 3.1,
            "ead2_diff": -26.0,
            "top20_diff": -0.3,
            "vocab_per_span_diff": -54.8,
            "f1": 0.939,
            "kappa": 0.611,
        },
        {
            "category": "Lacks Synthesis",
            "avg_span_len_diff": -14.56,
            "mattr_diff": -6.3,
            "ead2_diff": -37.2,
            "top20_diff": 30.2,
            "vocab_per_span_diff": -65.9,
            "f1": 0.960,
            "kappa": 0.108,
        },
    ]
    return pd.DataFrame(data)


def add_absolute_shift_columns(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        df[f"abs_{c}"] = df[c].abs()
    return df


def corr_report(
    df: pd.DataFrame,
    x_cols: List[str],
    y_cols: List[str],
) -> pd.DataFrame:
    rows = []

    for x in x_cols:
        for y in y_cols:
            xvals = df[x].to_numpy()
            yvals = df[y].to_numpy()

            if len(xvals) < 2:
                continue

            pearson_r, pearson_p = pearsonr(xvals, yvals)
            spearman_rho, spearman_p = spearmanr(xvals, yvals)

            rows.append(
                {
                    "x_metric": x,
                    "y_metric": y,
                    "pearson_r": pearson_r,
                    "pearson_p": pearson_p,
                    "spearman_rho": spearman_rho,
                    "spearman_p": spearman_p,
                    "n": len(df),
                }
            )

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(
            by=["y_metric", "spearman_rho"],
            ascending=[True, False],
            key=lambda s: s.abs() if s.name in {"pearson_r", "spearman_rho"} else s,
        )
    return out


def main() -> None:
    df = make_dataframe()

    synthetic_signed = [
        "avg_span_len_diff",
        "mattr_diff",
        "ead2_diff",
        "top20_diff",
        "vocab_per_span_diff",
    ]
    model_metrics = ["f1", "kappa"]

    print("\n=== Input data ===")
    print(df.to_string(index=False))

    df_abs = add_absolute_shift_columns(df, synthetic_signed)

    print("\n=== Absolute-shift data added ===")
    print(df_abs.to_string(index=False))

    print("\n=== Correlation using SIGNED synthetic shifts ===")
    signed_report = corr_report(df, synthetic_signed, model_metrics)
    print(signed_report.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    abs_cols = [f"abs_{c}" for c in synthetic_signed]

    print("\n=== Correlation using ABSOLUTE synthetic shifts ===")
    abs_report = corr_report(df_abs, abs_cols, model_metrics)
    print(abs_report.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    print(
        "\nInterpretation tip:\n"
        "- Signed shifts tell you whether over- or under-representation matters.\n"
        "- Absolute shifts tell you whether larger mismatch from real data is associated with worse performance.\n"
        "- With only 4 categories, treat p-values cautiously and use this mainly as supportive evidence."
    )


if __name__ == "__main__":
    main()


=== Input data ===
                category  avg_span_len_diff  mattr_diff  ead2_diff  top20_diff  vocab_per_span_diff    f1  kappa
                  Format              -7.10        -0.8      -20.4        -3.9                -47.8 0.929  0.788
Unsupported Claim (long)             -11.50        -0.8      -25.0        23.9                -49.3 0.495  0.209
               Coherence              31.60         3.1      -26.0        -0.3                -54.8 0.939  0.611
         Lacks Synthesis             -14.56        -6.3      -37.2        30.2                -65.9 0.960  0.108

=== Absolute-shift data added ===
                category  avg_span_len_diff  mattr_diff  ead2_diff  top20_diff  vocab_per_span_diff    f1  kappa  abs_avg_span_len_diff  abs_mattr_diff  abs_ead2_diff  abs_top20_diff  abs_vocab_per_span_diff
                  Format              -7.10        -0.8      -20.4        -3.9                -47.8 0.929  0.788                   7.10             0.8           20.4      